# 04 — Calibrated Data Inspector

Interactive time-series explorer for the calibrated daily files.  Select a date, choose which gas channels to show, and toggle between raw and calibrated traces.  Calibration windows (Feb 6, Feb 12) are shaded automatically when those dates are loaded.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
_BASE = Path('/uufs/chpc.utah.edu/common/home/lin-group24/agm/Mobile_SLV/Data/2026')
CAL_DIR    = _BASE / 'calibrated'
MERGED_DIR = _BASE / 'merged'

# Use calibrated if it exists, otherwise fall back to merged
DATA_DIR = CAL_DIR if CAL_DIR.exists() else MERGED_DIR
if DATA_DIR == MERGED_DIR:
    print('NOTE: calibrated/ not found — showing merged/ (no _cal columns).')
else:
    print(f'Loading from: {DATA_DIR}')

ALL_DATES = sorted([f.stem for f in DATA_DIR.glob('*.csv') if f.stem != '20260303'])
print(f'{len(ALL_DATES)} dates: {ALL_DATES}')

# ── Instrument colours ────────────────────────────────────────────────────────
INST_COLORS = {
    'Picarro':  '#1f77b4',
    'Ultra460': '#ff7f0e',
    'Ultra321': '#2ca02c',
    'Pico017':  '#d62728',
    'LGR':      '#9467bd',
}

# ── Calibration windows (UTC) ─────────────────────────────────────────────────
CAL_WINDOWS = {
    '20260212': [
        {'label': 'N2 zero', 'start': '2026-02-12 22:25:30', 'end': '2026-02-12 22:26:45',
         'color': 'rgba(168,216,234,0.4)', 'cert': 'CH4=0.0 ppm, C3H8=0.0'},
        {'label': 'NOAA',    'start': '2026-02-12 22:46:00', 'end': '2026-02-12 22:52:00',
         'color': 'rgba(184,240,184,0.4)', 'cert': 'CH4=2.001 ppm, C2H6=1.63 ppb'},
        {'label': 'Dil.1',   'start': '2026-02-12 22:57:00', 'end': '2026-02-12 22:58:00',
         'color': 'rgba(255,250,180,0.5)', 'cert': 'CH4=5.74 ppm, C3H8=1.0 ppm'},
        {'label': 'Dil.2',   'start': '2026-02-12 23:01:00', 'end': '2026-02-12 23:03:00',
         'color': 'rgba(255,215,0,0.4)',   'cert': 'CH4=11.47 ppm, C3H8=2.0 ppm'},
        {'label': 'Dil.3',   'start': '2026-02-12 23:03:30', 'end': '2026-02-12 23:04:30',
         'color': 'rgba(255,160,64,0.4)',  'cert': 'CH4=17.2 ppm, C3H8=3.0 ppm'},
        {'label': 'Dil.4',   'start': '2026-02-12 23:05:30', 'end': '2026-02-12 23:07:00',
         'color': 'rgba(255,96,48,0.4)',   'cert': 'CH4=28.88 ppm, C3H8=5.0 ppm'},
        {'label': 'Dil.5',   'start': '2026-02-12 23:08:00', 'end': '2026-02-12 23:09:00',
         'color': 'rgba(204,68,204,0.4)',  'cert': 'CH4=57.33 ppm, C3H8=10.0 ppm'},
    ],
    '20260206': [
        {'label': 'N2 zero', 'start': '2026-02-06 18:15:45', 'end': '2026-02-06 18:16:20',
         'color': 'rgba(168,216,234,0.4)', 'cert': 'CH4=0.0 ppm'},
        {'label': 'Dil.3',   'start': '2026-02-06 18:18:00', 'end': '2026-02-06 18:19:20',
         'color': 'rgba(255,160,64,0.4)',  'cert': 'CH4=17.2 ppm, C3H8=3.0 ppm'},
        {'label': 'Dil.2',   'start': '2026-02-06 18:20:00', 'end': '2026-02-06 18:21:30',
         'color': 'rgba(255,215,0,0.4)',   'cert': 'CH4=11.47 ppm, C3H8=2.0 ppm'},
        {'label': 'Dil.1',   'start': '2026-02-06 18:22:20', 'end': '2026-02-06 18:23:40',
         'color': 'rgba(255,250,180,0.5)', 'cert': 'CH4=5.74 ppm, C3H8=1.0 ppm'},
        {'label': 'Dil.4',   'start': '2026-02-06 18:27:00', 'end': '2026-02-06 18:28:30',
         'color': 'rgba(255,96,48,0.4)',   'cert': 'CH4=28.88 ppm, C3H8=5.0 ppm'},
        {'label': 'N2 zero', 'start': '2026-02-06 18:29:30', 'end': '2026-02-06 18:30:30',
         'color': 'rgba(168,216,234,0.4)', 'cert': 'CH4=0.0 ppm'},
    ],
}

GAS_ORDER = ['CH4', 'CO2', 'CO', 'C2H6', 'C3H8']


In [ ]:
def load_date(date_str):
    df = pd.read_csv(DATA_DIR / f'{date_str}.csv')
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])
    return df


def get_inst(col):
    for inst in INST_COLORS:
        if col.endswith(f'_{inst}') or col.endswith(f'_{inst}_cal'):
            return inst
    return None


def get_gas_groups(df):
    """Return {gas: [col, ...]} for all detectable measurement columns."""
    groups = {g: [] for g in GAS_ORDER}
    for col in df.columns:
        if col == 'TIMESTAMP':
            continue
        gas = col.split('_')[0]
        if gas in groups:
            groups[gas].append(col)
    # preserve a natural sort: raw before _cal, Picarro first within each gas
    def sort_key(c):
        order = list(INST_COLORS.keys())
        inst = get_inst(c) or 'z'
        return (0 if inst not in order else order.index(inst),
                1 if c.endswith('_cal') else 0)
    return {g: sorted(cols, key=sort_key) for g, cols in groups.items() if cols}


def col_style(col):
    """Return plotly line dict and opacity for a column."""
    inst  = get_inst(col) or 'Picarro'
    color = INST_COLORS.get(inst, '#888')
    is_cal = col.endswith('_cal')
    return dict(
        line=dict(color=color,
                  dash='dot' if is_cal else 'solid',
                  width=2.0 if is_cal else 1.2),
        opacity=1.0 if is_cal else 0.55,
    )


def build_figure(df, selected_gases, show_raw, show_cal, show_cal_windows, date_str):
    gas_groups = get_gas_groups(df)
    rows_to_plot = [g for g in GAS_ORDER
                    if g in selected_gases and g in gas_groups]

    if not rows_to_plot:
        fig = go.Figure()
        fig.update_layout(title='No channels selected.')
        return fig

    n = len(rows_to_plot)
    fig = make_subplots(
        rows=n, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=rows_to_plot,
    )

    legend_seen = set()
    for row_i, gas in enumerate(rows_to_plot, start=1):
        for col in gas_groups[gas]:
            is_cal = col.endswith('_cal')
            if is_cal and not show_cal:
                continue
            if not is_cal and not show_raw:
                continue

            valid = df['TIMESTAMP'].notna() & df[col].notna()
            if valid.sum() == 0:
                continue

            sty = col_style(col)
            inst = get_inst(col) or '?'
            suffix = ' (cal)' if is_cal else ''
            name = f'{inst}{suffix} [{gas}]'
            show_in_legend = name not in legend_seen
            legend_seen.add(name)

            fig.add_trace(
                go.Scattergl(
                    x=df.loc[valid, 'TIMESTAMP'],
                    y=df.loc[valid, col],
                    name=name,
                    legendgroup=name,
                    showlegend=show_in_legend,
                    hovertemplate=(
                        f'<b>{col}</b><br>'
                        '%{x|%H:%M:%S UTC}<br>'
                        '%{y:.4f}<extra></extra>'
                    ),
                    **sty,
                ),
                row=row_i, col=1,
            )

        # y-axis unit label from first column in group
        sample_col = gas_groups[gas][0]
        parts = sample_col.split('_')
        unit = parts[1] if len(parts) > 1 else ''
        fig.update_yaxes(title_text=unit, title_font_size=11, row=row_i, col=1)

    # ── Calibration window overlays ───────────────────────────────────────────
    if show_cal_windows and date_str in CAL_WINDOWS:
        for win in CAL_WINDOWS[date_str]:
            for row_i in range(1, n + 1):
                fig.add_vrect(
                    x0=win['start'], x1=win['end'],
                    fillcolor=win['color'],
                    layer='below', line_width=0,
                    row=row_i, col=1,
                )
            # Label annotation at top of figure
            t0 = pd.Timestamp(win['start'])
            t1 = pd.Timestamp(win['end'])
            mid = t0 + (t1 - t0) / 2
            fig.add_annotation(
                x=mid, xref='x', y=1.01, yref='paper',
                text=f"<b>{win['label']}</b><br><span style='font-size:8px'>{win['cert']}</span>",
                showarrow=False, font=dict(size=9),
                xanchor='center', yanchor='bottom',
                bgcolor='rgba(255,255,255,0.75)',
                bordercolor='#aaa', borderwidth=1,
            )

    # ── Layout ────────────────────────────────────────────────────────────────
    dt_str = f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}'
    src = 'calibrated' if DATA_DIR == CAL_DIR else 'merged (no cal)'
    fig.update_layout(
        height=max(320 * n + 80, 400),
        title=dict(text=f'SLV 2026 — {dt_str} UTC  [{src}]', font_size=14),
        legend=dict(
            orientation='v', x=1.01, y=1,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='#ccc', borderwidth=1,
            font=dict(size=10),
        ),
        margin=dict(l=65, r=190, t=70, b=50),
        hovermode='x unified',
        template='plotly_white',
    )
    fig.update_xaxes(tickformat='%H:%M', row=n, col=1,
                     title_text='UTC time', title_font_size=11)
    return fig


In [ ]:
# ── Controls ──────────────────────────────────────────────────────────────────

date_dd = widgets.Dropdown(
    options=ALL_DATES, value='20260212',
    description='Date:',
    style={'description_width': '40px'},
    layout=widgets.Layout(width='195px'),
)

gas_checks = {
    g: widgets.Checkbox(
        value=(g in ('CH4', 'C2H6', 'C3H8')),
        description=g, indent=False,
        layout=widgets.Layout(width='75px'),
    )
    for g in GAS_ORDER
}

raw_cb = widgets.Checkbox(
    value=True, description='Raw', indent=False,
    layout=widgets.Layout(width='65px'),
)
cal_cb = widgets.Checkbox(
    value=True, description='Calibrated', indent=False,
    layout=widgets.Layout(width='105px'),
)
cal_win_tb = widgets.ToggleButton(
    value=True, description='Cal windows', icon='check',
    button_style='info',
    layout=widgets.Layout(width='130px', height='30px'),
)

out = widgets.Output()

# ── Update callback ────────────────────────────────────────────────────────────

def update(_=None):
    selected = {g for g, cb in gas_checks.items() if cb.value}
    with out:
        clear_output(wait=True)
        try:
            df = load_date(date_dd.value)
        except FileNotFoundError:
            print(f'File not found for {date_dd.value} in {DATA_DIR}')
            return
        fig = build_figure(
            df,
            selected_gases=selected,
            show_raw=raw_cb.value,
            show_cal=cal_cb.value,
            show_cal_windows=cal_win_tb.value,
            date_str=date_dd.value,
        )
        fig.show()

# ── Wire observers ─────────────────────────────────────────────────────────────

for w in [date_dd, raw_cb, cal_cb, cal_win_tb] + list(gas_checks.values()):
    w.observe(update, names='value')

# ── Widget layout ──────────────────────────────────────────────────────────────

row1 = widgets.HBox([
    date_dd,
    widgets.HTML('&nbsp;&nbsp;'),
    cal_win_tb,
])
row2 = widgets.HBox(
    [widgets.Label('Gases:  ', layout=widgets.Layout(width='55px'))]
    + [gas_checks[g] for g in GAS_ORDER],
)
row3 = widgets.HBox([
    widgets.Label('Traces: ', layout=widgets.Layout(width='55px')),
    raw_cb, cal_cb,
])

panel = widgets.VBox(
    [row1, row2, row3],
    layout=widgets.Layout(
        border='1px solid #d0d0d0', padding='10px 14px',
        margin='0 0 10px 0', background_color='#fafafa',
    ),
)

display(panel, out)
update()


## Calibration window zoom-in

Load Feb 6 or Feb 12, zoom into the calibration sequence, and compare raw vs calibrated values against the known certified concentrations.

In [ ]:
# ── Quick-look: zoom in on one calibration date ───────────────────────────────
# Change CAL_DATE and GAS to inspect any window closely.

CAL_DATE = '20260212'    # '20260206' or '20260212'
GAS      = 'CH4'         # 'CH4', 'C2H6', or 'C3H8'

# Tight zoom: 10 min before first window to 5 min after last
windows = CAL_WINDOWS.get(CAL_DATE, [])
if not windows:
    print(f'No calibration windows defined for {CAL_DATE}')
else:
    t_start = pd.Timestamp(windows[0]['start'])  - pd.Timedelta('10min')
    t_end   = pd.Timestamp(windows[-1]['end'])   + pd.Timedelta('5min')

    df_cal = load_date(CAL_DATE)
    zoom   = df_cal[(df_cal['TIMESTAMP'] >= t_start) & (df_cal['TIMESTAMP'] <= t_end)].copy()

    gas_groups = get_gas_groups(zoom)
    gas_cols   = gas_groups.get(GAS, [])
    if not gas_cols:
        print(f'No {GAS} columns found for {CAL_DATE}')
    else:
        fig2 = go.Figure()
        for col in gas_cols:
            valid = zoom['TIMESTAMP'].notna() & zoom[col].notna()
            sty   = col_style(col)
            inst  = get_inst(col) or '?'
            suffix = ' (cal)' if col.endswith('_cal') else ''
            fig2.add_trace(go.Scattergl(
                x=zoom.loc[valid, 'TIMESTAMP'], y=zoom.loc[valid, col],
                name=f'{inst}{suffix}',
                hovertemplate='<b>%{fullData.name}</b><br>%{x|%H:%M:%S}<br>%{y:.4f}<extra></extra>',
                **sty,
            ))

        # Shade calibration windows
        for win in windows:
            fig2.add_vrect(
                x0=win['start'], x1=win['end'],
                fillcolor=win['color'], layer='below', line_width=0,
            )
            fig2.add_annotation(
                x=pd.Timestamp(win['start']) + (pd.Timestamp(win['end']) - pd.Timestamp(win['start'])) / 2,
                y=1.02, yref='paper',
                text=f"<b>{win['label']}</b><br><span style='font-size:9px'>{win['cert']}</span>",
                showarrow=False, font=dict(size=10),
                xanchor='center', yanchor='bottom',
                bgcolor='rgba(255,255,255,0.8)',
                bordercolor='#aaa', borderwidth=1,
            )

        dt_str = f'{CAL_DATE[:4]}-{CAL_DATE[4:6]}-{CAL_DATE[6:]}'
        fig2.update_layout(
            title=f'{GAS} — calibration sequence zoom  ({dt_str} UTC)',
            xaxis=dict(tickformat='%H:%M:%S', title='UTC time'),
            yaxis_title=gas_cols[0].split('_')[1] if gas_cols else '',
            hovermode='x unified',
            template='plotly_white',
            height=420,
            legend=dict(x=1.01, y=1, font_size=10, bgcolor='rgba(255,255,255,0.8)',
                        bordercolor='#ccc', borderwidth=1),
            margin=dict(l=65, r=170, t=70, b=50),
        )
        fig2.show()
